In [ ]:
# ============================================================
# Parallel directory copier  –  v5  (large-file + fd-safe)
# ============================================================
#
#  What's new vs v5
#  ─────────────────
#  • fd-safe buffered copy  –  explicit 64 MiB buffer, never calls
#    shutil internally, so chunk size is fully under our control
#  • Semaphore limits concurrent open file handles  →  no errno 24
#  • ulimit / setrlimit raised automatically at startup
#  • Worker count capped at 2 for large files (≥1 GB avg)
#    because the bottleneck is disk read, not parallelism
#  • Heartbeat recovery  –  alive resets to True when link returns
#  • Progress per file: prints MB written every 30 s for huge files
#  • Notebook status bars: global byte progress + completed directories

from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import os, hashlib, psutil, time, threading, stat

try:
    import resource  # Unix/Linux/macOS only; not available on Windows
except ImportError:
    resource = None
from dataclasses import dataclass
from typing import List, Dict, Optional, Tuple
import pandas as pd


# Optional notebook progress bars.
# If tqdm is not installed, the script still runs normally and prints text status.
try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None


In [ ]:
# ============================================================
# User settings
# ============================================================

SOURCE_DIRECTORIES = [
    r"\\from-storage\dir1",
    r"\\from-storage\dir2",
    r"\\from-storage\dir3"
]

DESTINATION_DIRECTORY = r"\\to_storage\dir1"

# "fast"  → file-count + total-size  (recommended for TB datasets)
# "hash"  → SHA-256 of every file    (definitive but slow on 250 GB files)
QUALITY_CHECK_MODE = "fast"

SKIP_ALREADY_COPIED  = True
PRESERVE_METADATA    = True

# Retry / network resilience
MAX_RETRIES            = 6      # per file
RETRY_BASE_DELAY       = 3.0   # seconds, doubles each attempt
HEARTBEAT_INTERVAL_SEC = 10

# Copy buffer – 64 MiB is optimal for large files on fast SMB links.
# Increase to 128 MiB if you have plenty of RAM and a dedicated link.
COPY_BUFFER_SIZE = 64 * 1024 * 1024   # 64 MiB

# Max simultaneous open file pairs (src+dst = 2 fds each).
# Keep well below the OS fd limit (typically 1024 on Linux, 512 on Windows).
# 6 pairs = 12 fds  →  safe even on restricted environments.
MAX_OPEN_FILE_PAIRS = 6

# Manual worker override (None = auto based on avg file size + network speed)
MAX_WORKERS_OVERRIDE = None


In [ ]:
# ============================================================
# Raise file-descriptor limit  (prevents errno 24 on Unix/Linux)
# ============================================================

def raise_fd_limit(target: int = 4096) -> tuple[int, int]:
    """
    Attempt to raise the process soft fd limit to *target*.
    Returns (old_soft, new_soft).

    On Windows the stdlib `resource` module is not available, so this is a
    no-op. The script still limits concurrent open files using `_fd_semaphore`.
    """
    if resource is None:
        print("  [INFO] resource module not available on Windows; skipping fd-limit raise.")
        return 0, 0

    try:
        soft, hard = resource.getrlimit(resource.RLIMIT_NOFILE)
        # Do not accidentally lower an already-high limit.
        desired = min(target, hard) if hard > 0 else target
        new_soft = max(soft, desired)
        if new_soft != soft:
            resource.setrlimit(resource.RLIMIT_NOFILE, (new_soft, hard))
        return soft, new_soft
    except (AttributeError, ValueError, OSError) as e:
        print(f"  [INFO] Could not raise fd limit: {e}")
        return 0, 0

old_fd, new_fd = raise_fd_limit(4096)
if new_fd:
    print(f"File-descriptor limit  :  {old_fd}  →  {new_fd}")
else:
    print("File-descriptor limit  :  not changed (Windows/no resource support — managed via semaphore)")

# Semaphore that caps concurrent open file pairs regardless of OS limit
_fd_semaphore = threading.Semaphore(MAX_OPEN_FILE_PAIRS)
print(f"Concurrent file pairs  :  max {MAX_OPEN_FILE_PAIRS}")


In [ ]:
# ============================================================
# Network speed benchmark
# ============================================================

def measure_network_speed_mbps(path: Path,
                                sample_mb: int = 64,
                                timeout: float = 45.0) -> float:
    """Write a probe file, measure write speed, return Mbit/s."""
    try:
        path.mkdir(parents=True, exist_ok=True)
        probe = path / ".speed_probe_tmp"
        chunk = b"\x00" * (1024 * 1024)
        total = sample_mb * 1024 * 1024

        t0 = time.perf_counter()
        written = 0
        with probe.open("wb", buffering=0) as f:   # unbuffered → true wire speed
            while written < total:
                f.write(chunk)
                written += len(chunk)
                if time.perf_counter() - t0 > timeout:
                    break

        elapsed = max(time.perf_counter() - t0, 0.001)
        probe.unlink(missing_ok=True)
        return (written * 8) / (elapsed * 1_000_000)

    except Exception as e:
        print(f"  [WARN] Speed benchmark failed ({e}). Assuming 100 Mbit/s.")
        return 100.0


dest_path = Path(DESTINATION_DIRECTORY).expanduser().resolve()

print("Benchmarking network write speed …  (64 MiB probe)")
NET_SPEED_MBPS = measure_network_speed_mbps(dest_path)
print(f"  {NET_SPEED_MBPS:,.0f} Mbit/s  "
      f"≈ {NET_SPEED_MBPS/8:,.0f} MB/s  "
      f"≈ {NET_SPEED_MBPS/1000:.2f} Gbit/s")


In [ ]:
# ============================================================
# System inspection  +  worker / chunk tuning
# ============================================================

def get_system_specs() -> Dict:
    cpu   = os.cpu_count() or 1
    ram   = psutil.virtual_memory()
    try:
        disk = psutil.disk_usage(str(dest_path.anchor))
        disk_free = disk.free / (1024**3)
    except Exception:
        disk_free = float("nan")
    return {
        "cpu_count"        : cpu,
        "ram_total_gb"     : ram.total     / (1024**3),
        "ram_available_gb" : ram.available / (1024**3),
        "disk_free_gb"     : disk_free,
        "net_speed_mbps"   : NET_SPEED_MBPS,
    }


def choose_workers(num_dirs: int, speed_mbps: float,
                   avg_file_gb: float) -> int:
    """
    For large files (≥ 1 GB avg) the bottleneck is sequential disk read.
    More workers fight for the same disk head  →  keep workers low.

    For small files on a fast network, parallelism helps amortise
    the per-file SMB negotiation overhead.
    """
    specs = get_system_specs()

    if avg_file_gb >= 10:
        io_limit = 2          # one read stream per physical disk
    elif avg_file_gb >= 1:
        io_limit = 4
    elif speed_mbps >= 4_000:
        io_limit = 16
    elif speed_mbps >= 800:
        io_limit = 8
    else:
        io_limit = 4

    ram_limit = max(1, int(specs["ram_available_gb"] // 0.25))
    return max(1, min(num_dirs, specs["cpu_count"], ram_limit, io_limit))


def choose_hash_chunk(speed_mbps: float) -> int:
    if speed_mbps >= 4_000:
        return 16 * 1024 * 1024
    elif speed_mbps >= 800:
        return 8  * 1024 * 1024
    else:
        return 4  * 1024 * 1024


specs = get_system_specs()
print("System specs:")
for k, v in specs.items():
    print(f"  {k}: {v:.2f}" if isinstance(v, float) else f"  {k}: {v}")

# We'll compute final workers after pre-flight (need avg file size)
HASH_CHUNK_SIZE = choose_hash_chunk(NET_SPEED_MBPS)
print(f"\nHash chunk : {HASH_CHUNK_SIZE // (1024*1024)} MiB")
print(f"Copy buffer: {COPY_BUFFER_SIZE  // (1024*1024)} MiB")


In [ ]:
# ============================================================
# Network heartbeat  (background thread, auto-recovers)
# ============================================================

class NetworkHeartbeat:
    """
    Writes a tiny probe to the destination every *interval* seconds.
    self.alive reflects the current reachability of the share —
    it can recover from True→False→True if the link comes back.
    """

    def __init__(self, dest: Path, interval: float = HEARTBEAT_INTERVAL_SEC):
        self.dest     = dest
        self.interval = interval
        self.alive    = True
        self._stop    = threading.Event()
        self._probe   = dest / ".heartbeat_probe"
        self._thread  = threading.Thread(target=self._run, daemon=True)

    def start(self):  self._thread.start()

    def stop(self):
        self._stop.set()
        self._thread.join(timeout=5)
        try: self._probe.unlink(missing_ok=True)
        except Exception: pass

    def _run(self):
        while not self._stop.wait(self.interval):
            try:
                self.dest.mkdir(parents=True, exist_ok=True)
                self._probe.write_bytes(b"hb")
                self._probe.unlink(missing_ok=True)
                if not self.alive:
                    print("  [HEARTBEAT] Network recovered — resuming.")
                self.alive = True
            except Exception as e:
                if self.alive:
                    print(f"  [HEARTBEAT] Network unreachable: {e}")
                self.alive = False

    def wait_for_recovery(self, poll: float = 5.0, timeout: float = 300.0):
        """Block until alive or timeout (seconds). Returns True if recovered."""
        t0 = time.time()
        while not self.alive:
            if time.time() - t0 > timeout:
                return False
            time.sleep(poll)
        return True


In [ ]:
# ============================================================
# Helpers
# ============================================================

@dataclass
class DirectoryStats:
    file_count       : int
    total_size_bytes : int

@dataclass
class CopyResult:
    source          : str
    destination     : str
    status          : str
    message         : str
    elapsed_seconds : float
    files_copied    : int = 0
    files_skipped   : int = 0
    bytes_copied    : int = 0


def get_directory_stats(directory: Path) -> DirectoryStats:
    count, size = 0, 0
    for root, _, files in os.walk(directory):
        for f in files:
            try:
                s = (Path(root) / f).stat()
                count += 1
                size  += s.st_size
            except FileNotFoundError:
                pass
    return DirectoryStats(count, size)


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        while chunk := f.read(HASH_CHUNK_SIZE):
            h.update(chunk)
    return h.hexdigest()


def get_directory_hashes(directory: Path) -> Dict[str, str]:
    out = {}
    for root, _, files in os.walk(directory):
        for f in files:
            p = Path(root) / f
            try:
                out[p.relative_to(directory).as_posix()] = sha256_file(p)
            except FileNotFoundError:
                pass
    return out


def quality_check(src: Path, dst: Path, mode: str = "fast") -> bool:
    if mode == "fast":
        ss, ds = get_directory_stats(src), get_directory_stats(dst)
        return ss.file_count == ds.file_count and ss.total_size_bytes == ds.total_size_bytes
    elif mode == "hash":
        return get_directory_hashes(src) == get_directory_hashes(dst)
    raise ValueError("QUALITY_CHECK_MODE must be 'fast' or 'hash'.")


In [ ]:
# ============================================================
# Notebook progress bars  (thread-safe, optional)
# ============================================================

_PROGRESS_BAR = None
_PROGRESS_LOCK = threading.Lock()


def set_progress_bar(bar):
    """Register the active tqdm progress bar used by worker threads."""
    global _PROGRESS_BAR
    with _PROGRESS_LOCK:
        _PROGRESS_BAR = bar


def progress_update(num_bytes: int):
    """Thread-safe update of the global byte progress bar."""
    if not num_bytes:
        return
    with _PROGRESS_LOCK:
        if _PROGRESS_BAR is not None:
            _PROGRESS_BAR.update(num_bytes)


def progress_close():
    """Close and unregister the global byte progress bar."""
    global _PROGRESS_BAR
    with _PROGRESS_LOCK:
        if _PROGRESS_BAR is not None:
            _PROGRESS_BAR.close()
            _PROGRESS_BAR = None


In [ ]:
# ============================================================
# Buffered file copy  (fd-safe, progress-aware)
# ============================================================

_PROGRESS_INTERVAL = 30   # seconds between progress prints for large files

def _buffered_copy(src: Path, dst: Path, preserve_meta: bool = True):
    """
    Copy src → dst using a fixed COPY_BUFFER_SIZE read/write loop.
    Uses the _fd_semaphore to cap concurrent open file handles.
    Prints progress every 30 s for files > 1 GB.
    """
    large = src.stat().st_size > 1024 ** 3   # >1 GB → show progress
    last_report = time.time()
    written_total = 0

    with _fd_semaphore:                       # ← caps open file pairs
        with src.open("rb") as fsrc, dst.open("wb") as fdst:
            while True:
                buf = fsrc.read(COPY_BUFFER_SIZE)
                if not buf:
                    break
                fdst.write(buf)
                chunk_len = len(buf)
                written_total += chunk_len
                progress_update(chunk_len)

                if large and time.time() - last_report >= _PROGRESS_INTERVAL:
                    pct = written_total / src.stat().st_size * 100
                    mb  = written_total / (1024 ** 2)
                    print(f"    … {src.name}  {mb:,.0f} MB  ({pct:.1f}%)")
                    last_report = time.time()

    if preserve_meta:
        try:
            s = src.stat()
            os.utime(dst, (s.st_atime, s.st_mtime))
        except Exception:
            pass


def copy_single_file(src: Path, dst: Path,
                     heartbeat: NetworkHeartbeat) -> Tuple[bool, bool]:
    """
    Returns (was_copied, was_skipped).
    Retries on transient errors, waits for heartbeat recovery.
    """
    # Resume check: same size → skip
    try:
        if dst.exists() and dst.stat().st_size == src.stat().st_size:
            progress_update(src.stat().st_size)
            return False, True
    except OSError:
        pass

    dst.parent.mkdir(parents=True, exist_ok=True)

    for attempt in range(1, MAX_RETRIES + 1):
        if not heartbeat.alive:
            print(f"    [WAIT] Network down — waiting up to 5 min for recovery …")
            if not heartbeat.wait_for_recovery(timeout=300):
                raise OSError("Network did not recover within 5 minutes.")

        try:
            _buffered_copy(src, dst, PRESERVE_METADATA)
            return True, False

        except OSError as exc:
            if exc.errno == 24:   # Too many open files — shouldn't happen now
                raise             # semaphore should prevent this; re-raise immediately
            if attempt == MAX_RETRIES:
                raise
            delay = RETRY_BASE_DELAY * (2 ** (attempt - 1))
            print(f"    [RETRY {attempt}/{MAX_RETRIES}] {src.name}: {exc}  "
                  f"(retry in {delay:.0f}s)")
            try: dst.unlink(missing_ok=True)
            except OSError: pass
            time.sleep(delay)

    return False, False


In [ ]:
# ============================================================
# Directory copy orchestrator
# ============================================================

def copy_directory(source_dir: str, destination_parent: str,
                   heartbeat: NetworkHeartbeat) -> CopyResult:
    t0  = time.time()
    src = Path(source_dir).expanduser().resolve()
    dst_root = Path(destination_parent).expanduser().resolve()
    dst = dst_root / src.name

    files_copied = files_skipped = bytes_copied = 0

    try:
        if not src.exists():
            return CopyResult(str(src), str(dst), "failed",
                              "Source does not exist.", time.time()-t0)
        if not src.is_dir():
            return CopyResult(str(src), str(dst), "failed",
                              "Source is not a directory.", time.time()-t0)

        dst_root.mkdir(parents=True, exist_ok=True)

        # Whole-directory skip if destination already complete
        if dst.exists() and SKIP_ALREADY_COPIED:
            try:
                ok = quality_check(src, dst, QUALITY_CHECK_MODE)
            except Exception as e:
                return CopyResult(str(src), str(dst), "failed",
                                  f"QC error on existing dest: {e}", time.time()-t0)
            if ok:
                try:
                    progress_update(get_directory_stats(src).total_size_bytes)
                except Exception:
                    pass
                return CopyResult(str(src), str(dst), "skipped",
                                  "Already exists and passed QC.", time.time()-t0)
            # Incomplete → fall through to resumable copy

        # File-by-file copy (resumable)
        for root, dirs, files in os.walk(src):
            rp       = Path(root)
            rel_root = rp.relative_to(src)
            dst_dir  = dst / rel_root

            # Preserve directory timestamps too
            dst_dir.mkdir(parents=True, exist_ok=True)

            for fname in files:
                src_f = rp / fname
                dst_f = dst_dir / fname

                try:
                    copied, skipped = copy_single_file(src_f, dst_f, heartbeat)
                    if copied:
                        files_copied  += 1
                        bytes_copied  += src_f.stat().st_size
                    elif skipped:
                        files_skipped += 1
                except Exception as e:
                    return CopyResult(str(src), str(dst), "failed",
                                      f"Failed on '{fname}': {e}",
                                      time.time()-t0,
                                      files_copied, files_skipped, bytes_copied)

        # Post-copy QC
        try:
            ok = quality_check(src, dst, QUALITY_CHECK_MODE)
        except Exception as e:
            return CopyResult(str(src), str(dst), "failed",
                              f"Post-copy QC error: {e}",
                              time.time()-t0, files_copied, files_skipped, bytes_copied)

        if not ok:
            return CopyResult(str(src), str(dst), "failed",
                              "Post-copy QC failed.",
                              time.time()-t0, files_copied, files_skipped, bytes_copied)

        return CopyResult(str(src), str(dst), "copied",
                          f"{files_copied} copied, {files_skipped} already present.",
                          time.time()-t0, files_copied, files_skipped, bytes_copied)

    except Exception as e:
        return CopyResult(str(src), str(dst), "failed", str(e),
                          time.time()-t0, files_copied, files_skipped, bytes_copied)


In [ ]:
# ============================================================
# Pre-flight checks  +  final worker count
# ============================================================

source_paths     = [Path(p).expanduser().resolve() for p in SOURCE_DIRECTORIES]
destination_root = Path(DESTINATION_DIRECTORY).expanduser().resolve()

valid_sources, missing_sources = [], []
for s in source_paths:
    (valid_sources if s.exists() and s.is_dir() else missing_sources).append(str(s))

print(f"Valid sources   : {len(valid_sources)}")
print(f"Missing/invalid : {len(missing_sources)}")
for p in missing_sources:
    print(f"  -  {p}")

if not valid_sources:
    raise RuntimeError("No valid source directories found. Check SOURCE_DIRECTORIES paths.")

# Aggregate stats
all_stats = [get_directory_stats(Path(s)) for s in valid_sources]
total_size_bytes = sum(st.total_size_bytes for st in all_stats)
total_files      = sum(st.file_count       for st in all_stats)
avg_file_gb      = (total_size_bytes / total_files / (1024**3)) if total_files else 0

try:
    free_gb = psutil.disk_usage(str(destination_root.anchor)).free / (1024**3)
except Exception:
    free_gb = float("inf")

print(f"\nTotal source size  : {total_size_bytes/(1024**3):.3f} GB")
print(f"Total files        : {total_files:,}")
print(f"Avg file size      : {avg_file_gb*1024:.1f} MB  ({avg_file_gb:.3f} GB)")
print(f"Destination free   : {free_gb:.3f} GB")

if total_size_bytes > free_gb * (1024**3):
    raise RuntimeError("Not enough free disk space at destination.")

if MAX_WORKERS_OVERRIDE is not None:
    MAX_WORKERS = int(MAX_WORKERS_OVERRIDE)
else:
    MAX_WORKERS = choose_workers(len(valid_sources), NET_SPEED_MBPS, avg_file_gb)

print(f"\nWorkers (tuned for avg {avg_file_gb*1024:.0f} MB files): {MAX_WORKERS}")
print("\nPre-flight OK — ready to copy.")


In [ ]:
# ============================================================
# Run
# ============================================================

results: List[CopyResult] = []

heartbeat = NetworkHeartbeat(destination_root)
heartbeat.start()
print(f"Heartbeat started  (every {HEARTBEAT_INTERVAL_SEC}s)\n")

start_all = time.time()

byte_bar = None
dir_bar = None

try:
    if tqdm is not None:
        byte_bar = tqdm(
            total=total_size_bytes,
            unit="B",
            unit_scale=True,
            unit_divisor=1024,
            desc="Copy/check progress",
            leave=True,
        )
        dir_bar = tqdm(
            total=len(valid_sources),
            desc="Directories completed",
            unit="dir",
            leave=True,
        )
        set_progress_bar(byte_bar)
    else:
        print("[INFO] tqdm not installed: progress bars disabled, text status still enabled.")
        set_progress_bar(None)

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        future_map = {
            executor.submit(copy_directory, src, DESTINATION_DIRECTORY, heartbeat): src
            for src in valid_sources
        }
        for future in as_completed(future_map):
            r = future.result()
            results.append(r)
            mb  = r.bytes_copied / (1024**2)
            thr = mb / r.elapsed_seconds if r.elapsed_seconds > 0 else 0
            print(f"[{r.status.upper():8s}]  "
                  f"{Path(r.source).name:<40s}  "
                  f"{r.elapsed_seconds:8.1f}s  "
                  f"{thr:7.1f} MB/s  |  {r.message}")
            if dir_bar is not None:
                dir_bar.update(1)
finally:
    heartbeat.stop()
    if dir_bar is not None:
        dir_bar.close()
    progress_close()

elapsed_all = time.time() - start_all
total_gb = sum(r.bytes_copied for r in results) / (1024**3)
print(f"\nTotal elapsed: {elapsed_all/60:.1f} min   |   Copied: {total_gb:,.2f} GB")


In [ ]:
# ============================================================
# Summary table
# ============================================================

cols = ["status", "source", "destination",
        "files_copied", "files_skipped",
        "bytes_copied_gb", "throughput_mb_s",
        "elapsed_seconds", "message"]

if results:
    df = pd.DataFrame([r.__dict__ for r in results])
    df["bytes_copied_gb"] = (df["bytes_copied"] / (1024**3)).round(3)
    df["throughput_mb_s"] = (
        (df["bytes_copied"] / (1024**2)) /
        df["elapsed_seconds"].replace(0, float("nan"))
    ).round(1)
else:
    # Keep the report/export cells from failing if the run produced no results.
    df = pd.DataFrame(columns=[
        "status", "source", "destination", "files_copied", "files_skipped",
        "bytes_copied", "bytes_copied_gb", "throughput_mb_s",
        "elapsed_seconds", "message"
    ])

df[cols]


In [ ]:
# ============================================================
# Save report
# ============================================================

report = destination_root / "copy_report_v4.csv"
df.to_csv(report, index=False)
print(f"Report saved → {report}")
